# Step 2: LoRA fine-tuning with BIPIA data added

**Capstone: Prompt-Injection Defense Evaluation, Notebook 10b**

Question: §5.11 LoRA-v1 (trained on deepset + neuralchemy + SPML, all direct injection) did NOT transfer to BIPIA's indirect injection (NB10 result: scores collapsed to a near-constant 0.81 cluster, no discriminative signal). Can we recover by training with BIPIA samples included?

## Two variants trained side by side

| Variant | Training data | Question it answers |
|---|---|---|
| **A. BIPIA-only LoRA** | BIPIA train split only (~560 rows) | Is the BIPIA distribution learnable by this architecture at all? |
| **B. Combined LoRA** | eval_set train (~3,182) + BIPIA train (~560) = 3,742 rows | Can a single LoRA cover both direct and indirect injection? |

Both variants use `ProtectAI/deberta-v3-base-prompt-injection-v2` as the base model (the §5.11 winner at 0.981 F1 on direct injection). Same LoRA recipe as §5.11: r=16, alpha=32, dropout=0.1, target_modules='all-linear', 3 epochs.

## Evaluation

Each variant evaluated on TWO held-out test sets to check for cross-distribution generalization and interference:

- BIPIA test split (~120 rows, ~7 clean / ~113 attack)
- eval_set test split (~680 rows, the same split used in §5.11)

Baseline comparisons reported alongside:
- Off-the-shelf ProtectAI DeBERTa (from §5.8 BIPIA results + §5.11 baseline metrics)
- §5.11 LoRA-from-ProtectAI (the best §5.11 variant; numbers loaded from `lora_metrics_extended.json`)

## Class imbalance handling

BIPIA is 94% positive (750 attacks / 50 clean). Without correction a classifier trained on BIPIA alone would trivially achieve 94% accuracy by predicting `INJECTION` for everything. Variant A uses class-weighted cross-entropy. Variant B doesn't strictly need it (combined data is ~65% positive) but uses the same weighting for consistency.

## Required Colab setup

- L4 GPU (T4 also works); A100 unnecessary at base model size
- High-RAM ON
- Mount Drive

## Required uploads to Drive (`MyDrive/capstone_lora/data/`)

Already in place from NB08 / NB10:
- `eval_set.parquet` (4,546 rows)
- `eval_set_splits.parquet` (70/15/15 split used in §5.11)
- `defense_a_full_eval_set.csv` (off-the-shelf ProtectAI on eval_set)
- `bipia_email_qa_prompts.csv` (800 BIPIA rows used in §5.8 / NB10)

Total wall time: ~25 min for both variants on L4.

## 1. Environment setup

In [1]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/capstone_lora')
DATA_DIR = DRIVE_ROOT / 'data'
RESULTS_DIR = DRIVE_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EVAL_SET_PATH = DATA_DIR / 'eval_set.parquet'
EVAL_SPLITS_PATH = DATA_DIR / 'eval_set_splits.parquet'
BIPIA_PROMPTS = DATA_DIR / 'bipia_email_qa_prompts.csv'
BIPIA_SPLITS_PATH = DATA_DIR / 'bipia_splits.parquet'

ADAPTER_DIR_A = DRIVE_ROOT / 'adapters' / 'lora_v2a_bipia_only'
ADAPTER_DIR_B = DRIVE_ROOT / 'adapters' / 'lora_v2b_combined'
ADAPTER_DIR_A.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR_B.mkdir(parents=True, exist_ok=True)

for p in [EVAL_SET_PATH, EVAL_SPLITS_PATH, BIPIA_PROMPTS]:
    print(f'  {p.name}: {"OK" if p.exists() else "MISSING upload first"}')

Mounted at /content/drive
  eval_set.parquet: OK
  eval_set_splits.parquet: OK
  bipia_email_qa_prompts.csv: OK


In [2]:
import os, sys, subprocess
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '--quiet', 'torchao'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers>=4.53', 'datasets', 'accelerate', 'scikit-learn',
    'peft', 'tqdm', 'sentencepiece'])
print('Packages installed.')

Packages installed.


In [3]:
import json
import time
import gc
import numpy as np
import pandas as pd
import torch
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    USE_BF16 = cc[0] >= 8
    USE_FP16 = not USE_BF16
    print(f'GPU: {torch.cuda.get_device_name(0)}, precision: {"bf16" if USE_BF16 else "fp16"}')
else:
    USE_BF16, USE_FP16 = False, False
    print('No GPU detected; training will be slow.')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

GPU: NVIDIA L4, precision: bf16


## 2. Load BIPIA and eval_set + create reproducible BIPIA split

Stratification key for BIPIA is `attack_category` directly (16 categories: 1 control + 15 attack categories), which preserves both class balance and category coverage in one stratification.

In [4]:
bipia = pd.read_csv(BIPIA_PROMPTS)
bipia = bipia.rename(columns={'full_prompt': 'prompt', 'is_attack': 'label'})
bipia['dataset'] = 'bipia'
print(f'BIPIA total: {len(bipia)}')
print(f'Class balance: {bipia["label"].value_counts().to_dict()}')
print(f'Attack categories: {bipia["attack_category"].nunique()}')

if BIPIA_SPLITS_PATH.exists():
    bipia_splits = pd.read_parquet(BIPIA_SPLITS_PATH)
    print(f'Loaded existing BIPIA splits from {BIPIA_SPLITS_PATH}')
else:
    train_val, test_ = train_test_split(
        bipia, test_size=0.15, random_state=SEED, stratify=bipia['attack_category']
    )
    train, val = train_test_split(
        train_val, test_size=0.15/0.85, random_state=SEED, stratify=train_val['attack_category']
    )
    train['split'] = 'train'
    val['split'] = 'val'
    test_['split'] = 'test'
    bipia_splits = pd.concat([train, val, test_], ignore_index=True)
    bipia_splits.to_parquet(BIPIA_SPLITS_PATH, index=False)
    print(f'Created BIPIA splits, saved to {BIPIA_SPLITS_PATH}')

print('\nBIPIA split sizes:')
print(bipia_splits.groupby('split')['label'].agg(['count', 'sum']))
print('\nClean controls per split:')
print(bipia_splits.groupby('split').apply(lambda d: (d['label']==0).sum()))

BIPIA total: 800
Class balance: {1: 750, 0: 50}
Attack categories: 16
Created BIPIA splits, saved to /content/drive/MyDrive/capstone_lora/data/bipia_splits.parquet

BIPIA split sizes:
       count  sum
split            
test     120  112
train    559  524
val      121  114

Clean controls per split:
split
test      8
train    35
val       7
dtype: int64


/tmp/ipykernel_5141/439562463.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(bipia_splits.groupby('split').apply(lambda d: (d['label']==0).sum()))


In [5]:
eval_splits = pd.read_parquet(EVAL_SPLITS_PATH)
print(f'eval_set splits loaded: {len(eval_splits)} rows total')
print('Split sizes (eval_set):')
print(eval_splits['split'].value_counts())
print('\nClass balance per split:')
print(eval_splits.groupby('split')['label'].agg(['count', 'sum', 'mean']).round(3))

eval_set splits loaded: 4546 rows total
Split sizes (eval_set):
split
train    3182
val       682
test      682
Name: count, dtype: int64

Class balance per split:
       count   sum   mean
split                    
test     682   361  0.529
train   3182  1687  0.530
val      682   362  0.531


## 3. Helpers: Wilson CIs, metrics, class-weighted Trainer

In [6]:
def wilson_ci(successes, n, alpha=0.05):
    if n == 0:
        return (0.0, 1.0)
    r = binomtest(int(successes), int(n))
    lo, hi = r.proportion_ci(confidence_level=1 - alpha, method='wilson')
    return float(lo), float(hi)

def per_class_metrics(y_true, y_pred, label_name=''):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    n_attack = int((y_true == 1).sum())
    n_clean = int((y_true == 0).sum())
    rec_ci = wilson_ci(tp, tp + fn) if (tp + fn) > 0 else (0.0, 0.0)
    prec_ci = wilson_ci(tp, tp + fp) if (tp + fp) > 0 else (0.0, 0.0)
    asr = 1.0 - (tp / max(n_attack, 1))
    far = fp / max(n_clean, 1)
    return {
        'label': label_name, 'n': len(y_true), 'n_pos': n_attack, 'n_neg': n_clean,
        'precision': float(p), 'precision_ci': prec_ci,
        'recall': float(r), 'recall_ci': rec_ci,
        'f1': float(f), 'accuracy': float(acc),
        'asr': float(asr), 'far': float(far),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    }

def print_metrics_table(metrics_list, title):
    print(f'\n=== {title} ===')
    print(f'{"Slice":<20} {"n":>5} {"n+":>5} {"Prec":>6} {"Recall":>7} {"F1":>6} {"ASR":>6} {"FAR":>6}')
    for m in metrics_list:
        print(f'{m["label"]:<20} {m["n"]:>5} {m["n_pos"]:>5} {m["precision"]:>6.3f} '
              f'{m["recall"]:>7.3f} {m["f1"]:>6.3f} {m["asr"]:>6.3f} {m["far"]:>6.3f}')

In [7]:
class WeightedTrainer(Trainer):
    """Trainer subclass that uses a fixed class-weight vector for cross-entropy.

    Necessary for Variant A (BIPIA-only): 94% positive class would otherwise let
    the model trivially predict 1 for everything.
    """
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            cw = torch.tensor(self.class_weights, device=logits.device, dtype=logits.dtype)
            loss_fn = torch.nn.CrossEntropyLoss(weight=cw)
        else:
            loss_fn = torch.nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', pos_label=1, zero_division=0)
    return {'accuracy': accuracy_score(labels, preds), 'precision': p, 'recall': r, 'f1': f1}

In [8]:
BASE_MODEL = 'ProtectAI/deberta-v3-base-prompt-injection-v2'
MAX_LENGTH = 512

def to_hf_dataset(df):
    return Dataset.from_pandas(
        df[['prompt', 'label']].rename(columns={'label': 'labels'}).reset_index(drop=True)
    )

def train_lora(train_df, val_df, run_label, adapter_save_path, num_epochs=3,
               learning_rate=2e-4, batch_size=16):
    """Train one LoRA variant. Returns (model, tokenizer, trainer, training_time_sec)."""
    print(f'\n=== Training {run_label} on {BASE_MODEL} ===')
    print(f'  train: {len(train_df)} rows, val: {len(val_df)} rows')
    print(f'  train class balance: {train_df["label"].value_counts().to_dict()}')

    classes = np.array([0, 1])
    weights = compute_class_weight('balanced', classes=classes, y=train_df['label'].values)
    print(f'  class weights: 0={weights[0]:.3f}, 1={weights[1]:.3f}')

    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=2,
        id2label={0: 'BENIGN', 1: 'INJECTION'},
        label2id={'BENIGN': 0, 'INJECTION': 1},
        ignore_mismatched_sizes=True,
    )
    cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
        lora_dropout=0.1, target_modules='all-linear', bias='none',
    )
    mdl = get_peft_model(mdl, cfg)
    mdl.to(device)
    n_trainable = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in mdl.parameters())
    print(f'  Trainable: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)')

    def tok_fn(batch):
        return tok(batch['prompt'], truncation=True, max_length=MAX_LENGTH, padding=False)
    tr_ds = to_hf_dataset(train_df).map(tok_fn, batched=True, remove_columns=['prompt'])
    vl_ds = to_hf_dataset(val_df).map(tok_fn, batched=True, remove_columns=['prompt'])
    coll = DataCollatorWithPadding(tokenizer=tok, padding=True, pad_to_multiple_of=8)

    args = TrainingArguments(
        output_dir=f'/content/lora_{run_label}',
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type='linear',
        logging_steps=25,
        eval_strategy='epoch',
        save_strategy='epoch',
        fp16=USE_FP16, bf16=USE_BF16,
        seed=SEED, report_to='none',
        load_best_model_at_end=True,
        metric_for_best_model='eval_f1',
        greater_is_better=True,
    )

    t0 = time.time()
    tr_obj = WeightedTrainer(
        model=mdl, args=args, train_dataset=tr_ds, eval_dataset=vl_ds,
        data_collator=coll, compute_metrics=compute_metrics_fn,
        class_weights=weights.tolist(),
    )
    tr_obj.train()
    elapsed = time.time() - t0
    print(f'  Trained in {elapsed/60:.1f} min')

    mdl.save_pretrained(adapter_save_path)
    tok.save_pretrained(adapter_save_path)
    print(f'  Adapter saved to {adapter_save_path}')
    return mdl, tok, tr_obj, elapsed

In [9]:
def evaluate_on(trainer, tok, test_df, slice_label, scores_col_name=None):
    """Run inference on a test dataframe; return metrics_dict + per-row preds/scores."""
    def tok_fn(batch):
        return tok(batch['prompt'], truncation=True, max_length=MAX_LENGTH, padding=False)
    te_ds = to_hf_dataset(test_df).map(tok_fn, batched=True, remove_columns=['prompt'])
    preds_obj = trainer.predict(te_ds)
    logits = preds_obj.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    y_pred = (probs > 0.5).astype(int)
    y_true = test_df['label'].values
    return {
        'metrics': per_class_metrics(y_true, y_pred, slice_label),
        'y_pred': y_pred.tolist(),
        'y_score': probs.tolist(),
    }

def cleanup_model(*objs):
    for o in objs:
        try: del o
        except: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Variant A: BIPIA-only LoRA

Diagnostic: is BIPIA learnable at all by this architecture? If yes (F1 > 0.85 on BIPIA test), the §5.11 transfer failure is purely a distribution-shift issue. If no, the recipe itself is wrong for indirect injection (would need different pooling, longer context, or different target modules).

In [10]:
bipia_train = bipia_splits[bipia_splits['split'] == 'train']
bipia_val = bipia_splits[bipia_splits['split'] == 'val']
bipia_test = bipia_splits[bipia_splits['split'] == 'test']

model_a, tok_a, trainer_a, time_a = train_lora(
    train_df=bipia_train,
    val_df=bipia_val,
    run_label='v2a_bipia_only',
    adapter_save_path=ADAPTER_DIR_A,
    num_epochs=3,
    learning_rate=2e-4,
    batch_size=16,
)


=== Training v2a_bipia_only on ProtectAI/deberta-v3-base-prompt-injection-v2 ===
  train: 559 rows, val: 121 rows
  train class balance: {1: 524, 0: 35}
  class weights: 0=7.986, 1=0.533


config.json:   0%|          | 0.00/994 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

Map:   0%|          | 0/559 [00:00<?, ? examples/s]

Map:   0%|          | 0/121 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.892610,0.668407,0.793388,0.958763,0.815789,0.881517
2,0.677274,0.917905,0.942149,0.942149,1.000000,0.970213
3,0.612158,0.654845,0.942149,0.957265,0.982456,0.969697


  Trained in 0.5 min
  Adapter saved to /content/drive/MyDrive/capstone_lora/adapters/lora_v2a_bipia_only


In [11]:
eval_test = eval_splits[eval_splits['split'] == 'test'].reset_index(drop=True)

result_a_bipia = evaluate_on(trainer_a, tok_a, bipia_test, 'BIPIA test (n=120)')
result_a_evalset = evaluate_on(trainer_a, tok_a, eval_test, 'eval_set test (n=682)')

metrics_a = [result_a_bipia['metrics'], result_a_evalset['metrics']]
for ds in ['deepset', 'neuralchemy', 'spml']:
    sub = eval_test[eval_test['dataset'] == ds].reset_index(drop=True)
    if len(sub) > 0:
        r = evaluate_on(trainer_a, tok_a, sub, f'  eval_set/{ds}')
        metrics_a.append(r['metrics'])

print_metrics_table(metrics_a, 'Variant A: BIPIA-only LoRA')

# Per-category breakdown on BIPIA test
print('\nVariant A: per-category recall on BIPIA test (attacks only)')
bipia_test_with_pred = bipia_test.reset_index(drop=True).copy()
bipia_test_with_pred['pred_a'] = result_a_bipia['y_pred']
bipia_test_with_pred['score_a'] = result_a_bipia['y_score']
cat_a = []
for cat in sorted(bipia_test_with_pred['attack_category'].unique()):
    mask = (bipia_test_with_pred['attack_category'] == cat) & (bipia_test_with_pred['label'] == 1)
    sub = bipia_test_with_pred[mask]
    if len(sub) > 0:
        recall = (sub['pred_a'] == 1).mean()
        mean_score = sub['score_a'].mean()
        print(f'  {cat:<32} n={len(sub):>2} recall={recall:>5.3f} mean_score={mean_score:>5.3f}')
        cat_a.append({'variant': 'A', 'category': cat, 'n': int(len(sub)),
                      'recall': float(recall), 'mean_score': float(mean_score)})

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/682 [00:00<?, ? examples/s]

Map:   0%|          | 0/82 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]


=== Variant A: BIPIA-only LoRA ===
Slice                    n    n+   Prec  Recall     F1    ASR    FAR
BIPIA test (n=120)     120   112  0.933   1.000  0.966  0.000  1.000
eval_set test (n=682)   682   361  0.910   0.925  0.918  0.075  0.103
  eval_set/deepset      82    30  1.000   0.700  0.824  0.300  0.000
  eval_set/neuralchemy   300   181  0.970   0.901  0.934  0.099  0.042
  eval_set/spml        300   150  0.838   1.000  0.912  0.000  0.193

Variant A: per-category recall on BIPIA test (attacks only)
  Base Encoding                    n= 7 recall=1.000 mean_score=0.851
  Business Intelligence            n= 7 recall=1.000 mean_score=0.835
  Conversational Agent             n= 8 recall=1.000 mean_score=0.844
  Emoji Substitution               n= 8 recall=1.000 mean_score=0.845
  Entertainment                    n= 8 recall=1.000 mean_score=0.859
  Information Dissemination        n= 7 recall=1.000 mean_score=0.851
  Language Translation             n= 8 recall=1.000 mean_score=0.

In [12]:
cleanup_model(model_a, trainer_a)
print(f'GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

GPU memory after cleanup: 0.79 GB


## 5. Variant B: combined (eval_set + BIPIA) LoRA

Deployment answer: can a single LoRA cover both direct and indirect injection? Combined train = eval_set train (~3,182 rows from deepset+neuralchemy+SPML) plus BIPIA train (~560 rows).

Two interference checks to watch for:
- BIPIA adding noise: does eval_set test F1 drop vs. §5.11 LoRA-from-ProtectAI (0.981)?
- eval_set drowning out BIPIA: does BIPIA test F1 stay close to Variant A?

In [13]:
eval_train = eval_splits[eval_splits['split'] == 'train'][['prompt', 'label', 'dataset']].copy()
eval_val = eval_splits[eval_splits['split'] == 'val'][['prompt', 'label', 'dataset']].copy()

bipia_train_min = bipia_train[['prompt', 'label']].copy()
bipia_train_min['dataset'] = 'bipia'
bipia_val_min = bipia_val[['prompt', 'label']].copy()
bipia_val_min['dataset'] = 'bipia'

combined_train = pd.concat([eval_train, bipia_train_min], ignore_index=True).sample(
    frac=1.0, random_state=SEED
).reset_index(drop=True)
combined_val = pd.concat([eval_val, bipia_val_min], ignore_index=True).reset_index(drop=True)

print(f'Combined train: {len(combined_train)} rows')
print(f'  by dataset:\n{combined_train["dataset"].value_counts()}')
print(f'  class balance: {combined_train["label"].value_counts().to_dict()}')
print(f'Combined val: {len(combined_val)} rows')

model_b, tok_b, trainer_b, time_b = train_lora(
    train_df=combined_train,
    val_df=combined_val,
    run_label='v2b_combined',
    adapter_save_path=ADAPTER_DIR_B,
    num_epochs=3,
    learning_rate=2e-4,
    batch_size=16,
)

Combined train: 3741 rows
  by dataset:
dataset
spml           1400
neuralchemy    1400
bipia           559
deepset         382
Name: count, dtype: int64
  class balance: {1: 2211, 0: 1530}
Combined val: 803 rows

=== Training v2b_combined on ProtectAI/deberta-v3-base-prompt-injection-v2 ===
  train: 3741 rows, val: 803 rows
  train class balance: {1: 2211, 0: 1530}
  class weights: 0=1.223, 1=0.846


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

  Trainable: 2,680,322 / 187,104,004 (1.43%)


Map:   0%|          | 0/3741 [00:00<?, ? examples/s]

Map:   0%|          | 0/803 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.090953,0.134131,0.971357,0.978858,0.972689,0.975764
2,0.097995,0.100040,0.975093,0.975000,0.983193,0.979079
3,0.122715,0.111884,0.975093,0.976987,0.981092,0.979036


  Trained in 4.8 min
  Adapter saved to /content/drive/MyDrive/capstone_lora/adapters/lora_v2b_combined


In [14]:
result_b_bipia = evaluate_on(trainer_b, tok_b, bipia_test, 'BIPIA test (n=120)')
result_b_evalset = evaluate_on(trainer_b, tok_b, eval_test, 'eval_set test (n=682)')

metrics_b = [result_b_bipia['metrics'], result_b_evalset['metrics']]
for ds in ['deepset', 'neuralchemy', 'spml']:
    sub = eval_test[eval_test['dataset'] == ds].reset_index(drop=True)
    if len(sub) > 0:
        r = evaluate_on(trainer_b, tok_b, sub, f'  eval_set/{ds}')
        metrics_b.append(r['metrics'])

print_metrics_table(metrics_b, 'Variant B: combined LoRA')

print('\nVariant B: per-category recall on BIPIA test (attacks only)')
bipia_test_with_pred['pred_b'] = result_b_bipia['y_pred']
bipia_test_with_pred['score_b'] = result_b_bipia['y_score']
cat_b = []
for cat in sorted(bipia_test_with_pred['attack_category'].unique()):
    mask = (bipia_test_with_pred['attack_category'] == cat) & (bipia_test_with_pred['label'] == 1)
    sub = bipia_test_with_pred[mask]
    if len(sub) > 0:
        recall = (sub['pred_b'] == 1).mean()
        mean_score = sub['score_b'].mean()
        print(f'  {cat:<32} n={len(sub):>2} recall={recall:>5.3f} mean_score={mean_score:>5.3f}')
        cat_b.append({'variant': 'B', 'category': cat, 'n': int(len(sub)),
                      'recall': float(recall), 'mean_score': float(mean_score)})

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/682 [00:00<?, ? examples/s]

Map:   0%|          | 0/82 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]


=== Variant B: combined LoRA ===
Slice                    n    n+   Prec  Recall     F1    ASR    FAR
BIPIA test (n=120)     120   112  0.933   1.000  0.966  0.000  1.000
eval_set test (n=682)   682   361  0.981   0.981  0.981  0.019  0.022
  eval_set/deepset      82    30  1.000   0.933  0.966  0.067  0.000
  eval_set/neuralchemy   300   181  0.973   0.978  0.975  0.022  0.042
  eval_set/spml        300   150  0.987   0.993  0.990  0.007  0.013

Variant B: per-category recall on BIPIA test (attacks only)
  Base Encoding                    n= 7 recall=1.000 mean_score=0.949
  Business Intelligence            n= 7 recall=1.000 mean_score=0.942
  Conversational Agent             n= 8 recall=1.000 mean_score=0.943
  Emoji Substitution               n= 8 recall=1.000 mean_score=0.941
  Entertainment                    n= 8 recall=1.000 mean_score=0.946
  Information Dissemination        n= 7 recall=1.000 mean_score=0.945
  Language Translation             n= 8 recall=1.000 mean_score=0.94

## 6. Headline comparison: §5.11 LoRA-v1 (NB10 result) vs Variant A vs Variant B

Three rows per evaluation slice (BIPIA test, eval_set test overall + by-dataset).

In [15]:
def make_row(label, metrics):
    return {
        'slice': label,
        'n': metrics['n'],
        'n+': metrics['n_pos'],
        'precision': round(metrics['precision'], 3),
        'recall': round(metrics['recall'], 3),
        'f1': round(metrics['f1'], 3),
        'asr': round(metrics['asr'], 3),
        'far': round(metrics['far'], 3),
    }

# Variant A
print('=== VARIANT A: BIPIA-only LoRA ===')
for m in metrics_a:
    print(make_row(m['label'], m))

print('\n=== VARIANT B: combined eval_set + BIPIA LoRA ===')
for m in metrics_b:
    print(make_row(m['label'], m))

print('\n=== Comparison takeaways ===')
a_bipia_f1 = result_a_bipia['metrics']['f1']
b_bipia_f1 = result_b_bipia['metrics']['f1']
b_evalset_f1 = result_b_evalset['metrics']['f1']
print(f'  Variant A BIPIA F1: {a_bipia_f1:.3f}')
print(f'  Variant B BIPIA F1: {b_bipia_f1:.3f}  (delta vs A: {b_bipia_f1 - a_bipia_f1:+.3f})')
print(f'  Variant B eval_set F1: {b_evalset_f1:.3f}  (delta vs §5.11 LoRA 0.981: {b_evalset_f1 - 0.981:+.3f})')

=== VARIANT A: BIPIA-only LoRA ===
{'slice': 'BIPIA test (n=120)', 'n': 120, 'n+': 112, 'precision': 0.933, 'recall': 1.0, 'f1': 0.966, 'asr': 0.0, 'far': 1.0}
{'slice': 'eval_set test (n=682)', 'n': 682, 'n+': 361, 'precision': 0.91, 'recall': 0.925, 'f1': 0.918, 'asr': 0.075, 'far': 0.103}
{'slice': '  eval_set/deepset', 'n': 82, 'n+': 30, 'precision': 1.0, 'recall': 0.7, 'f1': 0.824, 'asr': 0.3, 'far': 0.0}
{'slice': '  eval_set/neuralchemy', 'n': 300, 'n+': 181, 'precision': 0.97, 'recall': 0.901, 'f1': 0.934, 'asr': 0.099, 'far': 0.042}
{'slice': '  eval_set/spml', 'n': 300, 'n+': 150, 'precision': 0.838, 'recall': 1.0, 'f1': 0.912, 'asr': 0.0, 'far': 0.193}

=== VARIANT B: combined eval_set + BIPIA LoRA ===
{'slice': 'BIPIA test (n=120)', 'n': 120, 'n+': 112, 'precision': 0.933, 'recall': 1.0, 'f1': 0.966, 'asr': 0.0, 'far': 1.0}
{'slice': 'eval_set test (n=682)', 'n': 682, 'n+': 361, 'precision': 0.981, 'recall': 0.981, 'f1': 0.981, 'asr': 0.019, 'far': 0.022}
{'slice': '  eval_

## 7. Save metrics and per-row predictions

In [16]:
def serializable(m):
    out = dict(m)
    out['precision_ci'] = list(m['precision_ci'])
    out['recall_ci'] = list(m['recall_ci'])
    return out

summary = {
    'experiment': 'lora_v2_bipia_retraining',
    'base_model': BASE_MODEL,
    'lora_config': {'r': 16, 'alpha': 32, 'dropout': 0.1, 'target_modules': 'all-linear'},
    'training': {
        'epochs': 3, 'lr': 2e-4, 'batch_size': 16, 'seed': SEED,
        'class_weighted_loss': True,
        'time_min_variant_a': time_a / 60,
        'time_min_variant_b': time_b / 60,
    },
    'variant_a_bipia_only': {
        'train_size': len(bipia_train),
        'val_size': len(bipia_val),
        'metrics': [serializable(m) for m in metrics_a],
        'bipia_per_category': cat_a,
    },
    'variant_b_combined': {
        'train_size': len(combined_train),
        'val_size': len(combined_val),
        'metrics': [serializable(m) for m in metrics_b],
        'bipia_per_category': cat_b,
    },
}

metrics_path = RESULTS_DIR / 'lora_v2_metrics.json'
metrics_path.write_text(json.dumps(summary, indent=2))
print(f'Saved {metrics_path}')

per_row_path = RESULTS_DIR / 'lora_v2_bipia_test_preds.csv'
bipia_test_with_pred[['row_id', 'attack_category', 'label',
                      'pred_a', 'score_a', 'pred_b', 'score_b']].to_csv(per_row_path, index=False)
print(f'Saved {per_row_path}')

Saved /content/drive/MyDrive/capstone_lora/results/lora_v2_metrics.json
Saved /content/drive/MyDrive/capstone_lora/results/lora_v2_bipia_test_preds.csv


## 8. Robustness checks

Five checks test whether the §6 headline numbers are real measurements vs methodology artifacts. Each runs in seconds and either confirms robustness or flags a problem worth a caveat in the report.

| Check | What it tests | Healthy outcome |
|---|---|---|
| 8.1 Duplicates | Train/test prompt overlap | < 1% |
| 8.2 Length shortcut | Whether model uses prompt length as proxy for "attack" | corr(length, score \| attack) < 0.3 |
| 8.3 Score distribution | Whether model collapsed to a near-constant cluster (the NB10 LoRA-v1 failure mode) | Cohen d > 1.5 between clean and attack scores |
| 8.4 Confusion matrix | FP vs FN balance per slice | both rates reasonable |
| 8.5 Variant B interference | Whether adding BIPIA hurt direct-injection performance | delta F1 vs §5.11 ≥ -0.01 per dataset |

Run AFTER sections 4-5 so `result_a_*`, `result_b_*`, and `bipia_test_with_pred` are in scope.

In [17]:
# 8.1 Duplicate-prompt check across BIPIA splits and combined train.
# Memorization risk: identical prompts in train AND test inflate measured F1.
# BIPIA shares 50 base emails across all 16 categories so partial overlap is
# expected, but FULL-prompt overlap should be near zero after stratification.

for source_name, splits_df in [
    ('BIPIA', bipia_splits),
    ('combined', pd.concat([
        combined_train.assign(split='train'),
        combined_val.assign(split='val'),
        eval_test.assign(split='test'),
        bipia_test.assign(split='test'),
    ], ignore_index=True)),
]:
    train_set = set(splits_df[splits_df['split'] == 'train']['prompt'])
    test_set = set(splits_df[splits_df['split'] == 'test']['prompt'])
    overlap = train_set & test_set
    pct = 100 * len(overlap) / max(len(test_set), 1)
    flag = 'CLEAN' if pct < 1.0 else ('CAVEAT' if pct < 5.0 else 'WARNING')
    print(f'{source_name:<10}: {len(overlap)} of {len(test_set)} test prompts overlap train ({pct:.2f}%)  [{flag}]')

BIPIA     : 0 of 120 test prompts overlap train (0.00%)  [CLEAN]
combined  : 1 of 802 test prompts overlap train (0.12%)  [CLEAN]


In [18]:
# 8.2 Prompt-length-as-shortcut check on BIPIA test.
# BIPIA appends attack text to email bodies so attack prompts are systematically
# longer than clean prompts. If the LoRA learns 'long prompt -> injection' rather
# than attack content, headline F1 is an artifact. Test: does the score correlate
# with length WITHIN the attack class (where label is constant)?

bipia_test_lc = bipia_test_with_pred.copy()
bipia_with_prompt = bipia[['row_id', 'prompt']].set_index('row_id')
bipia_test_lc['prompt_len'] = bipia_test_lc['row_id'].map(bipia_with_prompt['prompt'].str.len())

len_stats = bipia_test_lc.groupby('label')['prompt_len'].agg(['mean', 'median', 'min', 'max']).round(0)
print('Prompt length (chars) by label on BIPIA test:')
print(len_stats)
attack_len = bipia_test_lc[bipia_test_lc['label']==1]['prompt_len'].mean()
clean_len = bipia_test_lc[bipia_test_lc['label']==0]['prompt_len'].mean()
print(f'mean(attack)/mean(clean) length ratio: {attack_len / max(clean_len, 1):.2f}x')

attacks = bipia_test_lc[bipia_test_lc['label']==1]
for col in ['score_a', 'score_b']:
    if len(attacks) > 5:
        corr = np.corrcoef(attacks['prompt_len'], attacks[col])[0, 1]
        flag = 'OK' if abs(corr) < 0.3 else ('CAVEAT' if abs(corr) < 0.5 else 'SHORTCUT RISK')
        print(f'  {col}: corr(length, score | label=1) = {corr:+.3f}  [{flag}]')

Prompt length (chars) by label on BIPIA test:
        mean  median  min   max
label                          
0      690.0   666.0  316  1178
1      668.0   699.0  317  1265
mean(attack)/mean(clean) length ratio: 0.97x
  score_a: corr(length, score | label=1) = +0.130  [OK]
  score_b: corr(length, score | label=1) = -0.063  [OK]


In [19]:
# 8.3 Score-distribution degeneracy check (the NB10 LoRA-v1 failure mode).
# NB10 LoRA-v1 on BIPIA: clean mean=0.808 std=0.011, attack mean=0.809 std=0.011
# -> nearly identical distributions, Cohen d ~0.1. The new variants must do
# materially better than that for any F1 / ASR / FAR number to be meaningful.

for variant_label, score_col in [('A (BIPIA-only)', 'score_a'), ('B (combined)', 'score_b')]:
    clean = bipia_test_with_pred[bipia_test_with_pred['label']==0][score_col]
    attack = bipia_test_with_pred[bipia_test_with_pred['label']==1][score_col]
    sep = abs(attack.mean() - clean.mean())
    pooled_std = ((clean.std()**2 + attack.std()**2) / 2) ** 0.5
    cohen_d = sep / max(pooled_std, 1e-6)
    print(f'Variant {variant_label} score distribution on BIPIA test:')
    print(f'  clean  (n={len(clean):>3}): mean={clean.mean():.3f}  std={clean.std():.3f}  range=[{clean.min():.3f}, {clean.max():.3f}]')
    print(f'  attack (n={len(attack):>3}): mean={attack.mean():.3f}  std={attack.std():.3f}  range=[{attack.min():.3f}, {attack.max():.3f}]')
    print(f'  separation |mean_attack - mean_clean| = {sep:.3f},  Cohen d = {cohen_d:.2f}')
    if cohen_d > 1.5:
        print(f'  HEALTHY: clean and attack distributions well separated.')
    elif cohen_d > 0.5:
        print(f'  MODEST separation; model has signal but room to improve.')
    else:
        print(f'  DEGENERATE: model has no discriminative signal (NB10 LoRA-v1 pattern).')
    print()

Variant A (BIPIA-only) score distribution on BIPIA test:
  clean  (n=  8): mean=0.834  std=0.023  range=[0.795, 0.859]
  attack (n=112): mean=0.844  std=0.018  range=[0.779, 0.875]
  separation |mean_attack - mean_clean| = 0.010,  Cohen d = 0.48
  DEGENERATE: model has no discriminative signal (NB10 LoRA-v1 pattern).

Variant B (combined) score distribution on BIPIA test:
  clean  (n=  8): mean=0.944  std=0.005  range=[0.939, 0.955]
  attack (n=112): mean=0.945  std=0.007  range=[0.934, 0.984]
  separation |mean_attack - mean_clean| = 0.001,  Cohen d = 0.13
  DEGENERATE: model has no discriminative signal (NB10 LoRA-v1 pattern).



In [20]:
# 8.4 Confusion matrices per slice (BIPIA test + eval_set test by dataset).
# Reveals whether errors are FP-dominated (over-flagging clean inputs, bad UX)
# or FN-dominated (missing real attacks, bad security). Asymmetry suggests the
# 0.5 default threshold should be retuned for the deployment use case.

def print_cm(y_true, y_pred, label):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / max(tn + fp, 1)
    fnr = fn / max(tp + fn, 1)
    print(f'  {label}:  TN={tn:>4} FP={fp:>4} FN={fn:>4} TP={tp:>4}   FPR={fpr:.3f} FNR={fnr:.3f}')

for variant_label, pred_col in [('A', 'pred_a'), ('B', 'pred_b')]:
    print(f'\n=== Variant {variant_label} confusion matrices ===')
    print_cm(bipia_test_with_pred['label'], bipia_test_with_pred[pred_col], 'BIPIA test       ')


=== Variant A confusion matrices ===
  BIPIA test       :  TN=   0 FP=   8 FN=   0 TP= 112   FPR=1.000 FNR=0.000

=== Variant B confusion matrices ===
  BIPIA test       :  TN=   0 FP=   8 FN=   0 TP= 112   FPR=1.000 FNR=0.000


In [21]:
# 8.5 Variant B interference check.
# Did adding BIPIA to the training mix degrade direct-injection performance?
# Compares Variant B per-dataset eval_set test F1 against the §5.11 LoRA-from-
# ProtectAI numbers (the §5.11 winner at overall 0.981 F1). If B materially
# underperforms §5.11 on any of deepset/neuralchemy/spml, the one-classifier
# approach is broken and the deployment story needs a router instead.

LORA_V1_PER_DATASET = {'overall': 0.981, 'deepset': 0.967, 'neuralchemy': 0.985, 'spml': 0.986}

print(f'{"slice":<15} {"variant_B_F1":>13} {"§5.11_F1":>10} {"delta":>8}  verdict')
for m in metrics_b:
    label = m['label'].strip()
    if 'eval_set test' in label:
        key = 'overall'
    elif 'eval_set/' in label:
        key = label.split('/')[-1].strip()
    else:
        continue
    if key not in LORA_V1_PER_DATASET:
        continue
    delta = m['f1'] - LORA_V1_PER_DATASET[key]
    verdict = 'NO INTERFERENCE' if delta >= -0.01 else ('minor degradation' if delta >= -0.03 else 'INTERFERENCE')
    print(f'  {key:<13} {m["f1"]:>13.3f} {LORA_V1_PER_DATASET[key]:>10.3f} {delta:>+8.3f}  {verdict}')

slice            variant_B_F1   §5.11_F1    delta  verdict
  overall               0.981      0.981   -0.000  NO INTERFERENCE
  deepset               0.966      0.967   -0.001  NO INTERFERENCE
  neuralchemy           0.975      0.985   -0.010  NO INTERFERENCE
  spml                  0.990      0.986   +0.004  NO INTERFERENCE


In [22]:
# 8.6 Save robustness check outcomes to the metrics JSON for the audit trail.

with open(RESULTS_DIR / 'lora_v2_metrics.json', 'r') as f:
    existing = json.load(f)

robustness = {}
for variant_label, score_col in [('a', 'score_a'), ('b', 'score_b')]:
    clean = bipia_test_with_pred[bipia_test_with_pred['label']==0][score_col]
    attack = bipia_test_with_pred[bipia_test_with_pred['label']==1][score_col]
    sep = float(abs(attack.mean() - clean.mean()))
    pooled_std = float(((clean.std()**2 + attack.std()**2) / 2) ** 0.5)
    cohen_d = sep / max(pooled_std, 1e-6)
    robustness[f'variant_{variant_label}_score_separation'] = {
        'clean_mean': float(clean.mean()), 'clean_std': float(clean.std()),
        'attack_mean': float(attack.mean()), 'attack_std': float(attack.std()),
        'separation': sep, 'cohen_d': cohen_d,
    }

existing['robustness_checks'] = robustness
with open(RESULTS_DIR / 'lora_v2_metrics.json', 'w') as f:
    json.dump(existing, f, indent=2)
print('Robustness check outcomes appended to lora_v2_metrics.json')

Robustness check outcomes appended to lora_v2_metrics.json


## 9. Interpretation guide for §5.11 write-up

### Variant A (BIPIA-only): is BIPIA learnable?

| Variant A BIPIA F1 | Reading |
|---|---|
| ≥ 0.90 | BIPIA is highly learnable. §5.11 LoRA-v1 transfer failure is purely a distribution shift problem, not a recipe limitation. |
| 0.70 - 0.90 | BIPIA is partially learnable with this recipe. Indirect injection signal is harder to extract but present. |
| < 0.70 | Recipe fails on indirect injection. Would need different pooling (max over content tokens?), longer context window, or different target modules. |

### Variant B (combined): does one LoRA work for both?

| Pattern | Reading |
|---|---|
| B BIPIA F1 ≈ A BIPIA F1 AND B eval_set F1 ≈ 0.981 | Combined training is the deployment answer. One LoRA, both distributions, no interference. Headline §5.11b finding. |
| B BIPIA F1 << A BIPIA F1 | eval_set drowns out BIPIA in the loss. Need oversampling of BIPIA in training, or per-distribution LoRA adapters. |
| B eval_set F1 << 0.981 | BIPIA introduces noise that degrades direct-injection performance. Need either careful loss weighting, or a router (two specialized LoRAs + a switcher). |
| Both drop | Neither distribution is learnable in the same head. Reconsider the architectural assumption that one classifier serves all. |

### Reading the robustness checks together with the headline

§5.11 write-up should always pair the headline F1 with the §8 outcomes:

- If §8.3 Cohen d is large but §8.1 duplicates > 5%, the F1 is partially memorization, not generalization.
- If §8.3 Cohen d is small (NB10 LoRA-v1 pattern), the F1 is meaningless regardless of value: the model has no signal.
- If §8.2 length correlation is high, the model is partially using length as a feature, narrowing the finding to the BIPIA-style "attack appended to email" composition.
- If §8.5 shows interference > 0.03 on any direct-injection dataset, drop the combined-LoRA claim and report Variants A and B as two separate models requiring a router.

### What to write in §5.11

The pre-existing §5.11 paragraph claims that LoRA closes the cross-dataset variance gap. This stretch result extends it to indirect injection or finds a real limit. Either outcome is publishable; the limit-finding is methodologically stronger because it sharpens scope.

### Files to download to repo

- `MyDrive/capstone_lora/results/lora_v2_metrics.json` → `results/lora_v2_metrics.json`
- `MyDrive/capstone_lora/results/lora_v2_bipia_test_preds.csv` → `results/lora_v2_bipia_test_preds.csv`
- `MyDrive/capstone_lora/data/bipia_splits.parquet` → `results/bipia_splits.parquet` (for reproducibility audit)
- Adapters (large, optional for storage): `lora_v2a_bipia_only/` and `lora_v2b_combined/`